In [1]:
##N=20000 or more
import os
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import math
from numpy import *
import pandas as pd
from scipy.optimize import curve_fit

In [ ]:
class structure_factor_model():
    
    def __init__(self,a1,z1,b2,z2,cx,cy,cz,gjf_name):
        self.ea=np.array([1.        , 0.0, 0.])
        self.eb=np.array([ 0.        , 1.0, 0.0])
        self.ez=np.array([0., 0., 1.])
        self.aa=a1*self.ea+z1*self.ez
        self.bb=b2*self.eb+z2*self.ez
        self.cc=cx*self.ea+cy*self.eb+cz*self.ez
        ## lattice vectors under optimization parameters
        self.a=self.aa
        self.b=self.bb
        self.c=self.cc
        ##############
        self.V=np.dot(np.cross(self.a,self.b),self.c)
        self.A=2*np.pi*np.cross(self.b,self.c)/self.V
        self.B=2*np.pi*np.cross(self.c,self.a)/self.V
        self.C=2*np.pi*np.cross(self.a,self.b)/self.V
        self.gjf_name=gjf_name
        self.N=4 #index range
        self.path_factor='\\path\\to\\XRD_pattern\\atom_scattering_factor_list\\'
        self.path_base='\\path\\to\\XRD_pattern\\opt_model\\'
        self.path_model=self.path_base+gjf_name

    def atom_factor(self,atom,t):
        with open(self.path_factor+atom+'.txt') as f:
            p=f.readlines()
        p=list(map(lambda x:float(x[:-2]),p))
        factor=p[0]*np.exp(-p[1]*(t**2))+p[2]*np.exp(-p[3]*(t**2))+p[4]*np.exp(-p[5]*(t**2))+p[6]*np.exp(-p[7]*(t**2))+p[8]
        if atom=='H':
            B_=8*(np.pi**2)*0.06
        else:
            B_=8*(np.pi**2)*0.05
        factor=factor*np.exp(-B_*(t**2))
        return factor

    def make_config(self):
        with open(self.path_model) as f:
            lines=f.readlines()
        lines_xyz=[]
        for line in lines:
            if line[:2]=='H ' or line[:2]=='C ' or line[:2]=='S ' or line[:2]=='O ':
                lines_xyz.append(line)
        xyz_list=list(map(lambda x : x.split(),lines_xyz))
        return xyz_list##

    def structure_factor(self,h,k,l):
        K=h*self.A+k*self.B+l*self.C ##inverse lattice vector
        K_norm=np.linalg.norm(K,ord=2)
        t=K_norm/(4*np.pi) #x=sin(theta)/wave_length=K/4π
        H_fac=self.atom_factor('H',t)
        C_fac=self.atom_factor('C',t)
        O_fac=self.atom_factor('O',t)
        S_fac=self.atom_factor('S',t)
        xyz_list=self.make_config()
        xyz_1=[]
        factor_list=[]
        for xyz in xyz_list:
            if xyz[0]=='H':
                atm_fac=H_fac
            elif xyz[0]=='C':
                atm_fac=C_fac
            elif xyz[0]=='O':
                atm_fac=O_fac
            elif xyz[0]=='S':
                atm_fac=S_fac
            else:
                continue
            x,y,z=list(map(float,xyz[1:4]))
            R=np.array([x,y,z])
            exp=np.exp(np.dot(K,R)*1j)
            factor_list.append(exp*atm_fac)
        return np.sum(factor_list)
    
    def make_csv(self,name_csv):
        N=self.N 
        df = pd.DataFrame(index=range(0,(2*N+1)**2),columns=range(-N,N+1))
        for l in range(-N,N+1):
            for k in range(-N,N+1):
                for h in range(-N,N+1):
                    stfc=self.structure_factor(h,k,l)
                    a=(2*N+1)*(k+N)+N+h
                    df[l][a]=stfc
        
        df.to_csv(self.path_base+name_csv+'.csv')
    

    def gen_theta(self,h,k,l,wav_len):
        K=np.linalg.norm(h*self.A+k*self.B+l*self.C, ord=2)
        k=2*np.pi/wav_len
        return math.degrees(np.arcsin(K/(2*k)))

    def LP(self,theta,mode='3D'):
        theta=np.radians(theta)
        if mode=='3D':
            return (1+(np.cos(2*theta))**2)/(((np.sin(theta))**2)*np.cos(theta))
        elif mode=='2D':
            return (1+(np.cos(2*theta))**2)/(np.sin(2*theta))

    def laue(self,x, aa, mu,N=20000):
        th=np.radians(x-mu)
        return aa*((np.sin(N*th)/np.sin(th))**2)

    #fwhm=0.1
    #sig=fwhm/(2*np.sqrt(2*np.log(2)))
    def peak_plot(self,I,theta_2):
        return self.laue(np.linspace(0,50,200000),aa=I,mu=theta_2)

    def peak_sum(self,theta_2_plot,I_list_plot):
        peak_sum=0
        for ind,I in enumerate(I_list_plot):
            peak=self.peak_plot(I,theta_2_plot[ind])
            peak_sum=peak_sum+peak
        return peak_sum
    
####################################################################   
    def make_df_1D(self,wav_len,name_csv):
        N=self.N
        df_stfac=pd.read_csv(self.path_base+name_csv+'.csv')
        I_list,theta_list,label_list=[],[],[]
        for h in range(-N,N+1):
            for k in range(-N,N+1):
                for l in range(-N,N+1):
                    theta=self.gen_theta(h,k,l,wav_len)
                    I_org=(abs(complex(df_stfac.at[(2*N+1)*(k+N)+N+h,str(l)][1:-1])))**2
                    I=I_org*self.LP(theta,mode='3D')
                    label='('+str(h)+' '+str(k)+' '+str(l)+')'
                    I_list.append(I)
                    theta_list.append(theta)
                    label_list.append(label)
        df_1D=pd.DataFrame(columns=['theta deg','label','I'])
        df_1D['theta deg']=theta_list
        df_1D['label']=label_list
        df_1D['I']=I_list
        self.df_1D=df_1D
    
    
    def powder_pattern(self,wav_len,name_csv,color='red'):
        N=self.N
        self.make_df_1D(wav_len,name_csv)
        theta_plot=np.array(self.df_1D['theta deg'])
        I_plot=np.array(self.df_1D['I'])
        label_plot=np.array(self.df_1D['label'])
        
        ind_sort=np.argsort(theta_plot)
        theta_plot_sort=2*np.sort(theta_plot) #2θ 
        I_plot_sort=I_plot[ind_sort]
        label_plot_sort=label_plot[ind_sort]
        
        I_plot_sort=[I_plot_sort[2*i] for i in range(int(((2*N+1)**3-1)/2))]
        theta_plot_sort=[theta_plot_sort[2*i] for i in range(int(((2*N+1)**3-1)/2))]
        label_plot_sort=[label_plot_sort[2*i] for i in range(int(((2*N+1)**3-1)/2))]

        I_plot_cor=self.peak_sum(theta_plot_sort[1:],I_plot_sort[1:])
        I_plot_cor=I_plot_cor/(0.0001*max(I_plot_cor[1:]))

        plt.rcParams["figure.figsize"] = 8,5
        plt.xticks(np.linspace(0,180,19))
        plt.yticks(np.linspace(0,10000,6))
        plt.rcParams["font.size"] = 16
        plt.xlim(0,50)
        plt.plot(np.linspace(0,50,200000),I_plot_cor,color=color,label=name_csv)
        plt.title('powder pattern')
        df_powder=pd.DataFrame(columns=['2theta','I','label'])
        df_powder['2theta']=theta_plot_sort
        df_powder['I']=np.array(I_plot_sort)*10000/max(I_plot_sort[1:])
        df_powder['label']=label_plot_sort
        df_powder.to_csv(self.path_base+name_csv+'_powder.csv')
        self.df_powder=df_powder

In [ ]:
class structure_factor_cif(structure_factor_model):
    
    def __init__(self,a_,b_,c_,alpha,beta,gamma,gjf_name):
        self.a=np.array([a_,0,0])
        self.b=np.array([b_*np.cos(np.radians(gamma)),b_*np.sin(np.radians(gamma)),0])
        self.c=np.array([c_*np.cos(np.radians(beta)),c_*(np.cos(np.radians(alpha))-np.cos(np.radians(beta))*np.cos(np.radians(gamma)))/np.sin(np.radians(gamma)),c_*np.sqrt(1-(np.cos(np.radians(beta)))**2-((np.cos(np.radians(alpha))-np.cos(np.radians(beta))*np.cos(np.radians(gamma)))/np.sin(np.radians(gamma)))**2)])
        ## lattice vectors under crystallographic parameters
        self.V=np.dot(np.cross(self.a,self.b),self.c)
        self.A=2*np.pi*np.cross(self.b,self.c)/self.V
        self.B=2*np.pi*np.cross(self.c,self.a)/self.V
        self.C=2*np.pi*np.cross(self.a,self.b)/self.V
        self.gjf_name=gjf_name
        self.N=4 #range of index
        self.path_factor='\\path\\to\\XRD_pattern\\atom_scattering_factor_list\\'
        self.path_base='\\path\\to\\XRD_pattern\\cryst\\'
        self.path_model=self.path_base+gjf_name
        
    def atom_factor(self,atom,t):
        return super().atom_factor(atom,t)
    
    def make_config(self):
        return super().make_config()
    
    def structure_factor(self,h,k,l):
        return super().structure_factor(h,k,l)
    
    def make_csv(self,name_csv):
        super().make_csv(name_csv)
        
    def gen_theta(self,h,k,l,wav_len):
        return super().gen_theta(h,k,l,wav_len)
    
    def LP(self,theta,mode='3D'):
        return super().LP(theta,mode='3D')
    
    def laue(self,x, aa, mu,N=20000):
        return super().laue(x, aa, mu,N=20000)
    
    def peak_plot(self,I,theta_2):
        return super().peak_plot(I,theta_2)
    
    def peak_sum(self,theta_2_plot,I_list_plot):
        return super().peak_sum(theta_2_plot,I_list_plot)
    
    def make_df_1D(self,wav_len,name_csv):
        super().make_df_1D(wav_len,name_csv)
        
    def powder_pattern(self,wav_len,name_csv,color='red'):
        super().powder_pattern(wav_len,name_csv,color)

In [ ]:
##opt=structure_factor_model(a1,z1,b2,z2,cx,cy,cz,'opt.xyz')
##opt.make_csv('C6_opt')
##opt.powder_pattern(1.54056,'opt',color='red')